In [5]:
import torch
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
model.eval()

for name, layer in model.named_children():
  print(name, "->", layer.__class__.__name__)

activations = {}

def get_activation(name):
  def hook(module, input, output):
    activations[name] = output.detach()
  return hook

for name, layer in model.named_children():
  layer.register_forward_hook(get_activation(name))

x = torch.rand(1, 3, 224, 224)

with torch.no_grad():
  model(x)

for name, act in activations.items():
  print(f"{name:10s} {tuple(act.shape)}")






conv1 -> Conv2d
bn1 -> BatchNorm2d
relu -> ReLU
maxpool -> MaxPool2d
layer1 -> Sequential
layer2 -> Sequential
layer3 -> Sequential
layer4 -> Sequential
avgpool -> AdaptiveAvgPool2d
fc -> Linear
conv1      (1, 64, 112, 112)
bn1        (1, 64, 112, 112)
relu       (1, 64, 112, 112)
maxpool    (1, 64, 56, 56)
layer1     (1, 64, 56, 56)
layer2     (1, 128, 28, 28)
layer3     (1, 256, 14, 14)
layer4     (1, 512, 7, 7)
avgpool    (1, 512, 1, 1)
fc         (1, 1000)
